In [ ]:
# import duckdb

# # Convert via a single SQL query
# duckdb.query("COPY (SELECT * FROM 'credit_card_default_ssb_dataset_50000.csv') TO 'credit_card_default_ssb_dataset_50000.parquet' (FORMAT PARQUET)")


In [25]:
from ssb_experimental import *
from utils.universal_data_loader import *

# Example Feature Groupings Setup
business_groups = {
"delinquency_vars":["dpd_avg_3m", "dpd_avg_6m", "dpd_avg_12m","max_dpd_12m"],            
"transaction_vars":["txn_count_avg_3m", "txn_count_avg_6m", "txn_count_avg_12m"],
"spend_vars":["spend_avg_3m", "spend_avg_6m", "spend_avg_12m"],
"repayment_vars":["payment_ratio_avg_3m","payment_ratio_avg_6m","payment_ratio_avg_12m"],
"card_utilization_vars":["utilization_avg_3m","utilization_avg_6m","utilization_avg_12m","utilization_max_12m"],
}

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000, 5000,500,100],
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="default_flag",
    n_jobs=-1,
    min_sample_size=5000,
    min_lift=1.5,
    top_n_vars=30,
    max_segments=10,
    enable_diversity=True,
    max_feature_reuse=1,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=business_groups,
    ignore_features=None,
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/Strategic_Segment_Builder/credit_card_default_ssb_dataset_50000.csv").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-04 04:41:39,230 | INFO     | [universal_data_loader.py:75] | Detecting local file format handler for extension: '.csv'...
2026-07-04 04:41:39,357 | INFO     | [ssb_experimental.py:128] | Feature group validation passed. (17 features mapped)
2026-07-04 04:41:39,358 | INFO     | [ssb_experimental.py:337] | Dynamic Grid Search Enabled: 8 total configurations.
2026-07-04 04:41:39,359 | INFO     | [ssb_experimental.py:358] | Iteration 1 | Remaining Volume: 50,000 | Base Rate: 7.28%
2026-07-04 04:41:53,990 | INFO     | [ssb_experimental.py:458] | Feature Usage Tracker Update -> 'utilization_avg_3m' used count = 1
2026-07-04 04:41:53,991 | INFO     | [ssb_experimental.py:458] | Feature Usage Tracker Update -> 'max_dpd_12m' used count = 1
2026-07-04 04:41:53,992 | INFO     | [ssb_experimental.py:458] | Feature Usage Tracker Update -> 'payment_ratio_avg_3m' used count = 1
2026-07-04 04:41:53,992 | INFO     | [ssb_experimental.py:473] | Segment 1 Captured (Size Floor: 100 | Lift Floor: 2

In [28]:
from prettytable import PrettyTable
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+---------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |     capture_rate    |        lift        |
+---------+-------------+---------------+--------------------+--------------------+---------------------+--------------------+
|   0.0   |   48738.0   |     3052.0    | 6.262054249251098  |       7.284        |        97.476       | 0.8596999243892227 |
|   1.0   |    151.0    |     123.0     | 81.45695364238411  |       7.284        |        0.302        | 11.182997479734228 |
|   2.0   |    105.0    |      72.0     | 68.57142857142857  |       7.284        |         0.21        | 9.413979759943516  |
|   3.0   |    133.0    |      79.0     |  59.3984962406015  |       7.284        |        0.266        |  8.15465352012651  |
|   4.0   |    118.0    |      70.0     | 59.32203389830509  |       7.284        | 0.23600000000000002 |  8.14

In [29]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   utilization_avg_3m=[0.85, inf) & max_dpd_12m=[33.62, inf) & payment_ratio_avg_3m=(-inf, 0.30)
SQL Filter: utilization_avg_3m >= 0.85 AND max_dpd_12m >= 33.62 AND payment_ratio_avg_3m < 0.30
--------------------------------------------------
Segment ID: 2
Raw Rule:   bureau_score=[582.50, 617.50) & dpd_avg_12m=[16.05, inf) & payment_ratio_avg_6m=(-inf, 0.37)
SQL Filter: (bureau_score >= 582.50 AND bureau_score < 617.50) AND dpd_avg_12m >= 16.05 AND payment_ratio_avg_6m < 0.37
--------------------------------------------------
Segment ID: 3
Raw Rule:   utilization_avg_6m=[0.74, inf) & missed_payments_last_6m=[2.50, inf) & dpd_avg_3m=[30.32, inf)
SQL Filter: utilization_avg_6m >= 0.74 AND missed_payments_last_6m >= 2.50 AND dpd_avg_3m >= 30.32
--------------------------------------------------
Segment ID: 4
Raw Rule:   risk_segment=<ArrowStringArray>
['HighRisk']
Length: 1, dtype: str & annual_income=[202928.00, 261094.00) & utilizatio

In [30]:
pd.DataFrame(final_eval)

,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,48738,3052.0,6.262054,7.284,97.476,0.859700
1,1,151,123.0,81.456954,7.284,0.302,11.182997
2,2,105,72.0,68.571429,7.284,0.210,9.413980
3,3,133,79.0,59.398496,7.284,0.266,8.154654
4,4,118,70.0,59.322034,7.284,0.236,8.144156
5,5,112,60.0,53.571429,7.284,0.224,7.354672
6,6,128,49.0,38.281250,7.284,0.256,5.255526
7,7,120,41.0,34.166667,7.284,0.240,4.690646
8,8,108,30.0,27.777778,7.284,0.216,3.813533
9,9,287,66.0,22.996516,7.284,0.574,3.157127


In [31]:
# Preparing the dataset for scoring and decile banding.
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
    SELECT *, 
    CASE WHEN utilization_avg_3m >= 0.85 AND max_dpd_12m >= 33.62 AND payment_ratio_avg_3m < 0.30 THEN 1 ELSE 0 END AS seg_1,
    CASE WHEN (bureau_score >= 582.50 AND bureau_score < 617.50) AND dpd_avg_12m >= 16.05 AND payment_ratio_avg_6m < 0.37 THEN 1 ELSE 0 END AS seg_2,
    CASE WHEN utilization_avg_6m >= 0.74 AND missed_payments_last_6m >= 2.50 AND dpd_avg_3m >= 30.32 THEN 1 ELSE 0 END AS seg_3,
    CASE WHEN risk_segment IN ('HighRisk') AND (annual_income >= 202928.00 AND annual_income < 261094.00) AND utilization_avg_12m >= 0.66 THEN 1 ELSE 0 END AS seg_4,
    CASE WHEN dpd_avg_6m >= 20.74 AND payment_ratio_avg_12m < 0.45 AND (spend_avg_3m >= 5510.50 AND spend_avg_3m < 7765.50) THEN 1 ELSE 0 END AS seg_5,
    CASE WHEN (utilization_max_12m >= 0.76 AND utilization_max_12m < 0.86) AND (cash_advance_amt_3m >= 12229.00 AND cash_advance_amt_3m < 16168.50) AND num_delinquent_accounts >= 2.50
 THEN 1 ELSE 0 END AS seg_6,
    CASE WHEN txn_count_avg_12m < 11.50 AND (spend_avg_6m >= 8366.50 AND spend_avg_6m < 11895.50) AND num_open_trades >= 8.50
 THEN 1 ELSE 0 END AS seg_7,                  
    CASE WHEN spend_avg_12m < 4538.50 AND (total_credit_limit >= 77194.00 AND total_credit_limit < 107939.50) AND txn_count_avg_6m < 10.87
 THEN 1 ELSE 0 END AS seg_8,       
    CASE WHEN txn_count_avg_3m < 9.71 AND age < 21.50
 THEN 1 ELSE 0 END AS seg_9,    

    FROM predicted
""").df()
conn.close()

In [33]:
scorer = StrategicSegmentScore(
    target_col="default_flag",
    primary_key="index",
    segment_cols=["seg_1", "seg_2", "seg_3", "seg_4", "seg_5", "seg_6", "seg_7", "seg_8", "seg_9"],
)

In [34]:
scorer.calculate_and_export_weights(predicted, export_path="scorecard_model_test1.json")

2026-07-04 04:46:21,013 | INFO     | [ssb_experimental.py:545] | Initializing DuckDB scorecard engine...
2026-07-04 04:46:21,155 | INFO     | [ssb_experimental.py:580] | Computing harmonic scorecard weights...
2026-07-04 04:46:21,156 | INFO     | [ssb_experimental.py:617] | Scoring training dataset via NumPy Linear Algebra engine...
2026-07-04 04:46:21,166 | INFO     | [ssb_experimental.py:634] | Dataset Zero-Inflation Rate: 92.72%
2026-07-04 04:46:21,167 | INFO     | [ssb_experimental.py:637] | High Zero-Inflation (>=80%). Isolating Active Population...
2026-07-04 04:46:21,168 | INFO     | [ssb_experimental.py:649] | Calibrating deciles across 1,262 target customers...


{'model_metadata': {'total_training_population': 50000,
  'active_scored_population': 1262,
  'active_population_pct': 2.52,
  'baseline_event_rate': 0.0728},
 'segment_weights': {'seg_1': {'weight': 73,
   'lift': 11.183,
   'response_rate': 0.8146,
   'capture_rate': 0.0338},
  'seg_2': {'weight': 40,
   'lift': 9.3872,
   'response_rate': 0.6838,
   'capture_rate': 0.022},
  'seg_3': {'weight': 41,
   'lift': 8.5569,
   'response_rate': 0.6233,
   'capture_rate': 0.025},
  'seg_4': {'weight': 39,
   'lift': 8.4709,
   'response_rate': 0.617,
   'capture_rate': 0.0239},
  'seg_5': {'weight': 37,
   'lift': 8.2179,
   'response_rate': 0.5986,
   'capture_rate': 0.0233},
  'seg_6': {'weight': 17,
   'lift': 5.6298,
   'response_rate': 0.4101,
   'capture_rate': 0.0157},
  'seg_7': {'weight': 10,
   'lift': 4.5047,
   'response_rate': 0.3281,
   'capture_rate': 0.0115},
  'seg_8': {'weight': 7,
   'lift': 4.0945,
   'response_rate': 0.2982,
   'capture_rate': 0.0093},
  'seg_9': {'weigh

In [36]:
scorecard_json_path = "scorecard_model_test1.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-07-04 04:46:38,986 | INFO     | [1890066070.py:2] | Loading scorecard model artifact from scorecard_model_test1.json...


In [37]:
model_artifact.get("decile_min_thresholds")

{'1': 73,
 '2': 41,
 '3': 40,
 '4': 39,
 '5': 21,
 '6': 17,
 '7': 14,
 '8': 14,
 '9': 10,
 '10': 7}

In [38]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 73
Segment: seg_2 | Weight: 40
Segment: seg_3 | Weight: 41
Segment: seg_4 | Weight: 39
Segment: seg_5 | Weight: 37
Segment: seg_6 | Weight: 17
Segment: seg_7 | Weight: 10
Segment: seg_8 | Weight: 7
Segment: seg_9 | Weight: 14


In [39]:
model_artifact.get("decile_min_thresholds")

{'1': 73,
 '2': 41,
 '3': 40,
 '4': 39,
 '5': 21,
 '6': 17,
 '7': 14,
 '8': 14,
 '9': 10,
 '10': 7}

In [40]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 73 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 40 ELSE 0 END AS seg_2_weighted,
    CASE WHEN seg_3 = 1 THEN 41 ELSE 0 END AS seg_3_weighted,
    CASE WHEN seg_4 = 1 THEN 39 ELSE 0 END AS seg_4_weighted,
    CASE WHEN seg_5 = 1 THEN 37 ELSE 0 END AS seg_5_weighted,
    CASE WHEN seg_6 = 1 THEN 17 ELSE 0 END AS seg_6_weighted,
    CASE WHEN seg_7 = 1 THEN 10 ELSE 0 END AS seg_7_weighted,
    CASE WHEN seg_8 = 1 THEN 7 ELSE 0 END AS seg_8_weighted,
    CASE WHEN seg_9 = 1 THEN 14 ELSE 0 END AS seg_9_weighted,
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted + seg_3_weighted + seg_4_weighted + seg_5_weighted+ 
                    seg_6_weighted+ seg_7_weighted+ seg_8_weighted+ seg_9_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=73 THEN 1
                    WHEN total_weight >= 41 THEN 2
                    WHEN total_weight >= 40 THEN 3
                    WHEN total_weight >= 39 THEN 4
                    WHEN total_weight >= 53 THEN 5
                    WHEN total_weight >= 21 THEN 6
                    WHEN total_weight >= 17 THEN 7
                    WHEN total_weight >= 14 THEN 8
                    WHEN total_weight >= 10 THEN 9
                    WHEN total_weight >= 7 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [41]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(default_flag) AS events, 
                    (SUM(default_flag)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()

In [42]:
scored

,decile_band,count,events,response_rate
0,0,48738,3052.0,6.262054
1,1,187,151.0,80.748663
2,2,131,76.0,58.015267
3,3,83,54.0,65.060241
4,4,111,66.0,59.459459
5,6,124,64.0,51.612903
6,7,124,49.0,39.516129
7,8,287,66.0,22.996516
8,9,114,37.0,32.456140
9,10,101,27.0,26.732673
